# Notebook 04: Contrastive Learning Foundations

## Why This Matters
SimCLR (Chen et al., 2020) unified a decade of self-supervised representation learning into one clean framework. The insight: you don't need labels to learn good representations — you just need a way to define which examples should be similar. Augmentation provides that definition for images, and the contrastive objective does the rest.

The same objective structure runs through MoCo, DINO, CLIP, and every modern embedding model. Understanding SimCLR means understanding all of them.

## Structure
1. The self-supervised setup — augmentation as supervision signal (narrative)
2. NT-Xent loss — the contrastive objective from scratch
3. The encoder and projection head
4. Data augmentation pipeline
5. SimCLR training loop
6. Representation quality: linear evaluation protocol
7. What the representations look like
8. Bridge to MoCo

## What You'll Learn
- Why augmentation-invariant representations are semantically meaningful
- How NT-Xent (normalized temperature-scaled cross entropy) works and why temperature matters
- The role of the projection head — why you train with it but evaluate without it
- Why batch size is so critical for contrastive learning quality
- The linear evaluation protocol as a clean measure of representation quality


## Background

### Why augmentation works as a supervision signal

Two augmented views of the same image should share the same semantic content — it's still a dog whether it's cropped, flipped, or color-jittered. A representation that is invariant to these perturbations has learned what's semantically stable across views, which is close to what "meaning" is.

SimCLR makes this precise:
1. Take an image x; apply two random augmentations to get x_i and x_j
2. Encode both to get representations z_i and z_j
3. Maximize agreement between z_i and z_j while minimizing agreement with all other images in the batch

The contrastive signal forces the encoder to find features that are stable across augmentations (semantic content) and ignore features that change (color, crop position, brightness).

### NT-Xent loss

For a batch of N images, you have 2N augmented views. For each view i, the positive is its pair j; all other 2(N-1) views are negatives.

```
L_i = -log [ exp(sim(z_i, z_j) / τ) / Σ_{k≠i} exp(sim(z_i, z_k) / τ) ]
```

Where sim is cosine similarity and τ (tau) is a temperature hyperparameter.

**Temperature τ:** Low temperature sharpens the distribution — the model gets large gradients from hard negatives and small gradients from easy ones. High temperature softens it — more uniform gradient signal. SimCLR uses τ=0.5; values typically range 0.05–1.0.

### The projection head

SimCLR projects the encoder output through a small MLP before applying the contrastive loss. Critically, the projection head is **discarded at evaluation** — downstream tasks use the encoder's output directly.

Why? The projection head lets the network dedicate some capacity to satisfying the contrastive objective (which cares about augmentation invariance) without forcing the encoder representations to be augmentation-invariant in ways that destroy task-relevant information.


In [ ]:
# %pip install -q torch torchvision numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
np.random.seed(42)

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Data Augmentation Pipeline

In [ ]:
class SimCLRAugmentation:
    """Two random augmentations of the same image — the positive pair."""
    def __init__(self, img_size=32, s=0.5):
        color_jitter = T.ColorJitter(0.8*s, 0.8*s, 0.8*s, 0.2*s)
        self.transform = T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomApply([color_jitter], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                        std=[0.2023, 0.1994, 0.2010]),
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)  # two views


class ContrastiveDataset(Dataset):
    """Wrap a dataset to return two augmented views per image."""
    def __init__(self, dataset, augmentation):
        self.dataset = dataset
        self.augmentation = augmentation

    def __len__(self): return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        xi, xj = self.augmentation(img)
        return xi, xj, label  # label only for evaluation, not training


# Load CIFAR-10 — small enough to run without a heavy GPU
base_transform = T.Compose([T.ToTensor()])
cifar_train = torchvision.datasets.CIFAR10(root="/tmp/cifar10", train=True,
                                           download=True, transform=base_transform)
cifar_test  = torchvision.datasets.CIFAR10(root="/tmp/cifar10", train=False,
                                           download=True, transform=T.Compose([
                                               T.ToTensor(),
                                               T.Normalize([0.4914,0.4822,0.4465],[0.2023,0.1994,0.2010])
                                           ]))

aug = SimCLRAugmentation(img_size=32)
contrastive_dataset = ContrastiveDataset(cifar_train, aug)
contrastive_loader  = DataLoader(contrastive_dataset, batch_size=128,
                                 shuffle=True, num_workers=0, drop_last=True)
test_loader = DataLoader(cifar_test, batch_size=256, shuffle=False, num_workers=0)

print(f"Contrastive dataset: {len(contrastive_dataset)} images")
print(f"Batch size: 128 → 256 augmented views per batch, 254 negatives per positive")

# Visualize augmentation pairs
xi, xj, _ = next(iter(contrastive_loader))
mean = torch.tensor([0.4914,0.4822,0.4465]).view(1,3,1,1)
std  = torch.tensor([0.2023,0.1994,0.2010]).view(1,3,1,1)
xi_vis = (xi[:4] * std + mean).clamp(0,1)
xj_vis = (xj[:4] * std + mean).clamp(0,1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    axes[0,i].imshow(xi_vis[i].permute(1,2,0).numpy())
    axes[0,i].axis("off")
    axes[0,i].set_title(f"View 1 (img {i})")
    axes[1,i].imshow(xj_vis[i].permute(1,2,0).numpy())
    axes[1,i].axis("off")
    axes[1,i].set_title(f"View 2 (img {i})")
axes[0,0].set_ylabel("xi", rotation=0, labelpad=30, fontsize=12)
axes[1,0].set_ylabel("xj", rotation=0, labelpad=30, fontsize=12)
plt.suptitle("SimCLR positive pairs — same image, different augmentations", fontsize=12)
plt.tight_layout()
plt.show()

## 2. NT-Xent Loss

In [ ]:
class NTXentLoss(nn.Module):
    """Normalized Temperature-scaled Cross Entropy Loss.
    
    For a batch of 2N views (N positive pairs), treats all 2(N-1) other views
    as negatives for each anchor.
    """
    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.tau = temperature

    def forward(self, z_i: torch.Tensor, z_j: torch.Tensor) -> torch.Tensor:
        """
        z_i, z_j: [N, D] — embeddings of the two augmented views
        """
        N = z_i.shape[0]

        # Normalize to unit sphere — cosine similarity via dot product
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)

        # Concatenate all 2N representations
        z = torch.cat([z_i, z_j], dim=0)  # [2N, D]

        # Compute all pairwise similarities
        sim = torch.mm(z, z.T) / self.tau  # [2N, 2N]

        # Mask out self-similarities (diagonal)
        mask = torch.eye(2*N, dtype=torch.bool, device=z.device)
        sim.masked_fill_(mask, -1e9)

        # Positive pairs: (i, N+i) and (N+i, i)
        # For view i, the positive is view N+i; for view N+i, the positive is view i
        targets = torch.cat([
            torch.arange(N, 2*N, device=z.device),   # positives for first half
            torch.arange(0, N,   device=z.device),   # positives for second half
        ])

        loss = F.cross_entropy(sim, targets)
        return loss


# Quick unit test
loss_fn = NTXentLoss(temperature=0.5)
z_i_test = torch.randn(8, 128)
z_j_test = torch.randn(8, 128)
loss_val = loss_fn(z_i_test, z_j_test)
print(f"NT-Xent loss (random embeddings): {loss_val.item():.4f}")
print(f"Expected: ~log(2N-1) = log({2*8-1}) = {np.log(2*8-1):.4f}  (uniform distribution baseline)")

## 3. Encoder and Projection Head

In [ ]:
class SimCLREncoder(nn.Module):
    """SimCLR: ResNet backbone + MLP projection head.
    
    The projection head maps encoder output to a lower-dimensional space
    where the contrastive loss is applied. It's discarded at evaluation.
    """
    def __init__(self, base_dim: int = 512, proj_dim: int = 128):
        super().__init__()

        # Backbone: ResNet-18 without the final classifier
        resnet = torchvision.models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()  # remove maxpool for CIFAR-32px
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])  # drop fc layer

        # Projection head: 2-layer MLP with ReLU
        self.projector = nn.Sequential(
            nn.Linear(base_dim, base_dim),
            nn.BatchNorm1d(base_dim),
            nn.ReLU(inplace=True),
            nn.Linear(base_dim, proj_dim),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Returns encoder representations (used for downstream tasks)."""
        return self.encoder(x).squeeze(-1).squeeze(-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Returns projected representations (used during contrastive training)."""
        h = self.encode(x)
        return self.projector(h)


model_simclr = SimCLREncoder(base_dim=512, proj_dim=128).to(device)
total_params = sum(p.numel() for p in model_simclr.parameters())
print(f"SimCLR model parameters: {total_params:,}")
print(f"  Encoder (ResNet-18): {sum(p.numel() for p in model_simclr.encoder.parameters()):,}")
print(f"  Projection head:     {sum(p.numel() for p in model_simclr.projector.parameters()):,}")
print()
# Forward pass check
x_test = torch.randn(4, 3, 32, 32, device=device)
h = model_simclr.encode(x_test)
z = model_simclr(x_test)
print(f"Encoder output shape:    {h.shape}  (batch × encoder_dim)")
print(f"Projection output shape: {z.shape}  (batch × proj_dim)")

## 4. SimCLR Training Loop

In [ ]:
def train_simclr(model, loader, epochs=5, lr=3e-4, temperature=0.5):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn   = NTXentLoss(temperature=temperature)
    losses    = []

    model.train()
    for epoch in range(epochs):
        epoch_loss, n_batches = 0.0, 0
        for xi, xj, _ in loader:
            xi, xj = xi.to(device), xj.to(device)
            zi = model(xi)  # projected embeddings
            zj = model(xj)
            loss = loss_fn(zi, zj)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches  += 1
        scheduler.step()
        avg = epoch_loss / n_batches
        losses.append(avg)
        print(f"Epoch {epoch+1}/{epochs}  loss: {avg:.4f}  lr: {scheduler.get_last_lr()[0]:.6f}")

    return losses


print("Training SimCLR for 5 epochs (demo — production uses 200-1000 epochs)...")
print("Loss should decrease from ~log(2N-1) ≈ 5.57 toward lower values.")
print()
losses = train_simclr(model_simclr, contrastive_loader, epochs=5)

plt.figure(figsize=(7,3))
plt.plot(losses, marker='o')
plt.xlabel("Epoch"); plt.ylabel("NT-Xent Loss")
plt.title("SimCLR Training Loss")
plt.axhline(y=np.log(2*128-1), color='r', linestyle='--', label='random baseline')
plt.legend(); plt.tight_layout(); plt.show()

## 5. Linear Evaluation Protocol

In [ ]:
def extract_features(model, loader, device):
    """Extract encoder representations (not projected) for linear evaluation."""
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device)
            h = model.encode(imgs)
            features.append(h.cpu().numpy())
            labels.append(targets.numpy())
    return np.concatenate(features), np.concatenate(labels)


# Create a simple evaluation loader for training set (no augmentation)
eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914,0.4822,0.4465],[0.2023,0.1994,0.2010])
])
cifar_train_eval = torchvision.datasets.CIFAR10("/tmp/cifar10", train=True,
                                                 download=False, transform=eval_transform)
train_eval_loader = DataLoader(cifar_train_eval, batch_size=256, shuffle=False, num_workers=0)

print("Extracting features from encoder (projection head discarded)...")
train_feats, train_labels = extract_features(model_simclr, train_eval_loader, device)
test_feats,  test_labels  = extract_features(model_simclr, test_loader, device)

print(f"Train features: {train_feats.shape}")
print(f"Test features:  {test_feats.shape}")

# Train a linear classifier on frozen representations
scaler = StandardScaler()
train_feats_scaled = scaler.fit_transform(train_feats)
test_feats_scaled  = scaler.transform(test_feats)

clf = LogisticRegression(max_iter=500, C=0.1, random_state=42)
clf.fit(train_feats_scaled, train_labels)
acc = accuracy_score(test_labels, clf.predict(test_feats_scaled))

print(f"\nLinear evaluation accuracy (5 epochs): {acc:.3f}")
print(f"Random baseline: 0.100")
print(f"SimCLR (1000 epochs, ResNet-50): ~0.933")
print()
print("The linear probe is a clean measure: if a linear classifier works on frozen features,")
print("the encoder has learned semantically structured representations without labels.")

## 6. Why Batch Size Matters So Much

Each batch of N images provides 2(N-1) negatives per positive. Larger batches = harder, more diverse negatives = better learning signal.

```
N=64:   126 negatives per positive
N=256:  510 negatives per positive
N=4096: 8190 negatives per positive  (original SimCLR paper)
```

This is why SimCLR originally required 512 TPUs and 4096 batch size. MoCo (Notebook 05) solves this with a memory queue — you get thousands of negatives without needing a huge batch.


## 7. Bridge to MoCo

SimCLR has two fundamental constraints:
1. **Large batch requirement** — quality scales with batch size; 4096 is optimal but needs significant compute
2. **Both views must fit in memory simultaneously** — gradient flows through both encoders, doubling memory

**MoCo** (He et al., 2020) solves both with two ideas:
- A **momentum encoder** for the key (positive/negative) representations — updated slowly via exponential moving average, not gradients
- A **queue** of recent key embeddings — thousands of negatives without a large batch

You get the diversity of a large batch (queue size = 65536 in the original paper) at the memory cost of a small batch. The trade-off is the queue representations are slightly stale — which is why the momentum update keeps them consistent.

Notebook 05 implements MoCo from scratch.


## Summary

| Concept | Detail |
|---------|--------|
| Positive pair | Two augmented views of the same image |
| NT-Xent | Cross entropy over cosine similarities scaled by temperature τ |
| Temperature τ | Lower = sharper gradients from hard negatives; ~0.5 is standard |
| Projection head | MLP trained with contrastive loss; **discarded at evaluation** |
| Linear evaluation | Frozen encoder + linear classifier = clean representation quality metric |
| Batch size scaling | More negatives per positive = better training signal = needs large GPU |

**Next:** Notebook 05 — Momentum Contrast (MoCo). Same contrastive objective, but with a momentum encoder and key queue that makes it feasible without massive batch sizes.
